In [ ]:
"""
Name: Module 7 Toy Classification Tree.py
Course: Data Preparation and Analysis
Created Date: September 30, 2023
Author: Ming-Long Lam, Ph.D.g
Organization: Illinois Institute of Technology
"""

In [ ]:
import matplotlib.pyplot as plt
import numpy
import pandas
import sys

In [ ]:
import itertools

In [ ]:
# Set some options for printing all the columns
numpy.set_printoptions(precision = 10, threshold = sys.maxsize)
numpy.set_printoptions(linewidth = numpy.inf)

In [ ]:
pandas.set_option('display.max_columns', None)
pandas.set_option('display.expand_frame_repr', False)
pandas.set_option('max_colwidth', None)

In [ ]:
pandas.options.display.float_format = '{:,.10f}'.format

In [ ]:
def plog2p (proportion):
   '''A function that calculates the value p * log2(p).
   If p > 0.0, then result = p * log2(p) where log2() is the base 2 logarithm function
   If p == 0.0, then result = 0.0
   Otherwise, then result is NaN

   Argument:
   1. proportion: a value between 0.0 and 1.0

   Output:
   1. result: the p * log2(p) value
   '''

   if (0.0 < proportion and proportion < 1.0):
      result = proportion * numpy.log2(proportion)
   elif (proportion == 0.0 or proportion == 1.0):
      result = 0.0
   else:
      result = numpy.nan

   return (result)

In [ ]:
def NodeEntropy (nodeCount):
   '''A function that calculate the entropy of a node given a row of counts.
   
   Argument:
   1. nodeCount: a Pandas Series of the counts of target values of the node
   
   Output:
   1. nodeTotal: the sum of counts of the node
   2. nodeEntropy: the entropy of the node.
   '''

   nodeTotal = numpy.sum(nodeCount)
   nodeProportion = nodeCount / nodeTotal
   nodeEntropy = - numpy.sum(nodeProportion.apply(plog2p))

   return (nodeTotal, nodeEntropy)

In [ ]:
def EntropyNominalSplit (target, catPredictor, splitList, debug = 'N'):
   '''A function that calculates the split entropy for splitting a categorical predictor into
   two groups by a split value

   Argument:
   1. target: a Pandas Series of target values
   2. catPredictor: a Pandas Series of predictor values
   3. splitList: a list of values for splitting the predictor values into two groups
   4. debug: a Y/N flag to show debugging information

   Output:
   1. nNode: number of nodes
   2. splitEntropy: the split entropy value
   '''

   branch_indicator = numpy.where(catPredictor.isin(splitList), 'LEFT', 'RIGHT')
   xtab = pandas.crosstab(index = branch_indicator, columns = target, margins = False, dropna = True)
   if (debug == 'Y'):
      print('Target: ', target)
      print('Predictor: ', catPredictor)
      print('Split Value: ', splitList)
      print('Split Crosstabulation:')
      print(xtab)

   nNode = 0
   splitEntropy = 0.0
   tableTotal = 0.0
   for idx, row in xtab.iterrows():
      nNode = nNode + 1
      rowTotal, rowEntropy = NodeEntropy(row)
      tableTotal = tableTotal + rowTotal
      splitEntropy = splitEntropy + rowTotal * rowEntropy

   splitEntropy = splitEntropy / tableTotal
  
   return(nNode, splitEntropy)

In [ ]:
trainData = pandas.read_excel('C:\\IIT\\Data Preparation and Analysis\\Data\\PepperSaltSugar.xlsx')

In [ ]:
# Univariate frequency bar charts
fig, (ax1, ax2) = plt.subplots(nrows = 1, ncols = 2, dpi = 200, sharey = True, figsize = (14,6))
fig.tight_layout()
ufreq_A = trainData['A'].astype('category').value_counts()
ax1.bar(ufreq_A.index, ufreq_A, color = 'seagreen')
ax1.set_xlabel('A')
ax1.set_ylabel('Number of Observations')
ax1.yaxis.grid(True)

In [ ]:
ufreq_B = trainData['B'].astype('category').value_counts()
ax2.bar(ufreq_B.index, ufreq_B, color = 'dodgerblue')
ax2.set_xlabel('B')
ax2.set_ylabel('')
ax2.yaxis.grid(True)

In [ ]:
plt.show()

In [ ]:
# Extract the columns for easier use
A = trainData['A']
B = trainData['B']

In [ ]:
crossTable = pandas.crosstab(index = A, columns = B, margins = True, dropna = True)   
print(crossTable)

In [ ]:
# Root node entropy
node_total, root_entropy = NodeEntropy (ufreq_B)
print('Total Count: ', node_total)
print('Root Node Entropy: ', root_entropy)

In [ ]:
# Calculate the entropy of all the possible splits
category_A = set(ufreq_A.index)
n_category = len(category_A)

In [ ]:
print('Number of Categories =', n_category)
print('Categories: \n', category_A)
print('Number of Possible Splits = ', (2**(n_category-1) - 1))

In [ ]:
def takeEntropy(s):
    return s[2]

In [ ]:
split_summary = []
for size in range(1, n_category//2 + 1):
    comb_size = itertools.combinations(category_A, size)
    for item in list(comb_size):
        left_branch = list(item)
        right_branch = list(category_A.difference(left_branch))
        nNode, splitEntropy = EntropyNominalSplit (B, A, left_branch, debug = 'N')
        if (nNode > 1):
            split_summary.append([left_branch, right_branch, splitEntropy])

In [ ]:
# Determine the split that yields the lowest splitEntropy
split_summary.sort(key = takeEntropy, reverse = False)

In [ ]:
print('=== Optimal Split ===')
print(' Left Branch Set: ', split_summary[0][0])
print('Right Branch Set: ', split_summary[0][1])
print('   Split Entropy: ', split_summary[0][2])

In [ ]:
df = pandas.DataFrame(split_summary)